In [3]:
print('Radha!')

Radha!


In [4]:
# Tool
from langchain_core.tools import tool

@tool
def get_customer(customer_id: int) -> dict:
    """Get customer information by customer ID."""

    customers = {
        101: {
            "name": "Rahul",
            "plan": "Premium"
        },
        102: {
            "name": "Priya",
            "plan": "Basic"
        }
    }

    return customers.get(
        customer_id,
        {"error": "Customer not found"}
    )

In [5]:
# Middleware
from langchain.agents.middleware import wrap_tool_call


@wrap_tool_call
def logging_middleware(request, handler):

    print("\n[MIDDLEWARE]")
    print("Tool:", request.tool_call["name"])
    print("Args:", request.tool_call["args"])

    response = handler(request)

    print("[MIDDLEWARE] Tool execution completed")

    return response

In [6]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from dotenv import  load_dotenv

load_dotenv()


llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

agent = create_agent(
    model=llm,
    tools=[get_customer],
    middleware=[logging_middleware]
)

In [7]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Get information about customer 101."
        }
    ]
})


[MIDDLEWARE]
Tool: get_customer
Args: {'customer_id': 101}
[MIDDLEWARE] Tool execution completed
